<a href="https://colab.research.google.com/github/RDGopal/IB9AU-2026/blob/main/FT5_The_App.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The App
This notebook loads a fine-tuned Qwen2-VL model from a zip file and launches a simple Gradio web application for car damage assessment. Students can upload a photo of a damaged car and receive an AI-generated damage analysis.

## Before You Begin
You will need the file **`qwen_finetuned.zip`** provided by your instructor.

**Steps to upload it:**
1. Click the **folder icon** in the left sidebar of Colab to open the file browser
2. Click the **upload icon** (arrow pointing up) and select `qwen_finetuned.zip`
3. Wait for the upload to complete (the file will appear in the browser)
4. Then run the cells below in order

### Step 1 – Install Dependencies

Install the libraries needed to run the model and the app.

In [ ]:
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install",
    "accelerate>=1.1.0",
    "qwen-vl-utils",
    "bitsandbytes>=0.43.0",
    "peft>=0.13.0",
    "gradio",
    "-q"], check=True)

print("✅ Dependencies installed.")
print("⚠️  Go to Runtime → Restart session, then continue from Step 2.")

### Step 2 – Unzip the Model

This cell verifies the zip file is intact and then extracts it.

> ⚠️ **If you get a `FileNotFoundError`:** the zip hasn't been uploaded yet. Use the folder icon in the left sidebar to upload it.

> ⚠️ **If you get a `BadZipFile` / `RuntimeError`:** the upload was interrupted and the file is incomplete. The error message will guide you through deleting and re-uploading it. **Tip:** large files upload more reliably via Google Drive than via direct Colab upload — if uploads keep cutting out, copy the zip to your Drive and use `drive.mount()` instead.

In [ ]:
import zipfile
import os

zip_path    = "qwen_finetuned.zip"
extract_path = "qwen_finetuned"

# Check the zip exists
if not os.path.exists(zip_path):
    raise FileNotFoundError(
        f"Could not find '{zip_path}'. "
        "Please upload it using the Colab file browser (folder icon → upload arrow) and re-run this cell."
    )

# Check the zip is not corrupted before extracting
print(f"Verifying {zip_path} ({os.path.getsize(zip_path)/1024/1024:.1f} MB)...")
try:
    with zipfile.ZipFile(zip_path, "r") as z:
        bad = z.testzip()   # Returns None if all files are OK
        if bad:
            raise zipfile.BadZipFile(f"Corrupt file inside zip: {bad}")
except zipfile.BadZipFile as e:
    raise RuntimeError(
        f"\n\n❌ The zip file appears to be corrupted or incomplete: {e}\n\n"
        "This usually means the Colab upload was interrupted before it finished.\n"
        "Please try again:\n"
        "  1. Delete the incomplete file: open the file browser (folder icon),\n"
        "     right-click qwen_finetuned.zip → Delete\n"
        "  2. Re-upload qwen_finetuned.zip (wait until 100% complete)\n"
        "  3. Re-run this cell\n\n"
        "Tip: If uploads keep failing, try Google Drive instead:\n"
        "  from google.colab import drive\n"
        "  drive.mount('/content/drive')\n"
        "  Then copy the zip from Drive rather than uploading directly."
    ) from e

# Only extract if not already done
if not os.path.exists(extract_path):
    print(f"Extracting...")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(".")
    print(f"✅ Extracted to: ./{extract_path}/")
else:
    print(f"✅ '{extract_path}' folder already exists — skipping extraction.")

# Confirm expected files are present
required_files = ["adapter_config.json"]
missing = [f for f in required_files if not os.path.exists(os.path.join(extract_path, f))]
if missing:
    raise ValueError(f"Zip extracted but missing expected files: {missing}. Check the zip is correct.")

print("Model files verified. Ready to load.")

### Step 3 – Load the Fine-Tuned Model

We load the base Qwen2-VL model in 4-bit quantisation (to save GPU memory), then overlay the fine-tuned LoRA adapters from the extracted folder.

- **Base model** — the original Qwen2-VL-2B-Instruct weights from Hugging Face
- **LoRA adapters** — the small fine-tuned weights your instructor trained, stored in `qwen_finetuned/`
- **Processor** — handles image resizing and text tokenisation before passing inputs to the model

In [ ]:
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

assert torch.cuda.is_available(), "No GPU detected! Go to Runtime → Change runtime type → T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")

adapter_path = "qwen_finetuned"

# 1. Configure 4-bit quantisation
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

# 2. Load the base model (device_map omitted to avoid accelerate version conflicts)
print("Loading base model... (this may take a minute)")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct",
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
)

# 3. Load the processor
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

# 4. Overlay the fine-tuned LoRA adapters
print("Loading fine-tuned adapters...")
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

print("✅ Model loaded and ready.")

### Step 4 – Launch the App

This cell defines the inference logic and launches a Gradio web interface.

- Upload any photo of a damaged car using the interface
- The model will analyse it and return a list of damage types it detects
- The `share=True` argument creates a temporary public URL so the app can be accessed from any browser (valid for 72 hours)

In [ ]:
import gradio as gr
from PIL import Image

def process_vision_info(messages):
    """Extracts PIL image objects from the Qwen message format."""
    image_inputs = []
    for message in messages:
        if "content" in message and isinstance(message["content"], list):
            for item in message["content"]:
                if item.get("type") == "image":
                    image_inputs.append(item["image"])
    return image_inputs if image_inputs else None, None  # No video inputs in this app

def analyze_claim(image_filepath):
    """Runs inference on an uploaded image and returns the damage assessment."""
    if image_filepath is None:
        return "Please upload an image to analyse."

    try:
        pil_image = Image.open(image_filepath).convert("RGB")
    except Exception as e:
        return f"Error opening image: {e}"

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": pil_image},
                {"type": "text", "text": (
                    "Analyse the image and list all distinct damage types visible on the car, "
                    "separated by commas. "
                    "Examples: 'scratch', 'dent', 'broken window', 'flat tire', "
                    "'paint chip', 'deformation', 'cracked headlight'."
                )}
            ]
        }
    ]

    text_input = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text_input],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to("cuda")

    generated_ids = model.generate(**inputs, max_new_tokens=200)
    generated_ids_trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    result = processor.batch_decode(
        generated_ids_trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False
    )[0]

    return result

# Launch the Gradio interface
interface = gr.Interface(
    fn=analyze_claim,
    inputs=gr.Image(type="filepath", label="Upload Damaged Car Photo"),
    outputs=gr.Textbox(label="AI Damage Assessment", lines=10),
    title="FinTech Auto-Claims Prototype",
    description=(
        "Upload a photo of a damaged car. "
        "The fine-tuned AI model will identify the types of damage present."
    ),
    examples=[]   # You can add example image paths here if desired
)

interface.launch(debug=False, share=True)

## Summary

This notebook loads a fine-tuned Qwen2-VL model from a zip file provided by the instructor and deploys it as a simple interactive web app using Gradio.

The overall pipeline is:
1. **Upload** `qwen_finetuned.zip` → **Unzip** → **Load** base model + LoRA adapters → **Run** Gradio app

### Taking It Further: Hugging Face Hub

For long-term sharing and deployment, the instructor can push the model to the [Hugging Face Hub](https://huggingface.co/) with one command:
```python
trainer.model.push_to_hub("your-username/qwen2-vl-car-damage")
processor.push_to_hub("your-username/qwen2-vl-car-damage")
```
Students can then load it directly by name — no zip file needed:
```python
model = PeftModel.from_pretrained(base_model, "your-username/qwen2-vl-car-damage")
```
This avoids the upload step entirely and gives you permanent, versioned model storage.